In [1]:
import ee 
from RadGEEToolbox import GenericCollection, LandsatCollection, get_palette
# import GEE_UBM
from GEE_UBM import InputCollections, build_model_ready_collection, OriginalUBMRun, ModifiedUBM1Run, check_merged_collection
import geemap
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
import os

In [2]:
service_account = 'localpythonscripts@ut-gee-ugs-bsf-dev.iam.gserviceaccount.com'
credentials = ee.ServiceAccountCredentials(service_account, 'C:\\Users\\mradwin\\ut-gee-ugs-bsf-dev-53dcc5d729e0.json')
ee.Initialize(credentials=credentials)

In [10]:
UT_boundary = ee.FeatureCollection("projects/ut-gee-ugs-bsf-dev/assets/Utah_Regional_Boundary").geometry()

big_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/big_creek').geometry()
blacks_fork = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/blacks_fork').geometry()
castle_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/castle_creek').geometry()
farmington_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/farmington_creek').geometry()
mammoth_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/mammoth_creek').geometry()
muddy_creek = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/muddy_creek').geometry()
weber_river = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/weber_river').geometry()
west_canyon = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/west_canyon').geometry()
whiterocks_uinta = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/UBM_Validation_Areas/whiterocks_uinta').geometry()

basin_geometries_dict = {
    "big_creek": big_creek,
    "blacks_fork": blacks_fork,
    "castle_creek": castle_creek,
    "farmington_creek": farmington_creek,
    "mammoth_creek": mammoth_creek,
    "muddy_creek": muddy_creek,
    "weber_river": weber_river,
    "west_canyon": west_canyon,
    "whiterocks_uinta": whiterocks_uinta
}
basins_list = [big_creek, blacks_fork, castle_creek, farmington_creek, mammoth_creek, muddy_creek, weber_river, west_canyon, whiterocks_uinta]
basins_names = ["big_creek", "blacks_fork", "castle_creek", "farmington_creek", "mammoth_creek", "muddy_creek", "weber_river", "west_canyon", "whiterocks_uinta"]

In [5]:
geomaterials = ee.FeatureCollection('projects/ut-gee-ugs-bsf-dev/assets/GeoMaterial_USGS_CNGM_Utah')
m_s_to_m_month_conversion = 30.44 * 24 * 3600  # Convert m/s to m/month


# RESTORING VERSION 9 (The "Natural Physics" Baseline)
k_values = {
    # --- ALPINE DRIVERS (Weber/Uinta Headwaters) ---
    # Restoring the higher values that made Weber work (-11%)
    'Glacial till': 2.5e-5 * m_s_to_m_month_conversion,         # V11 was 0.5e-5 (Too low)
    'Metasedimentary rock': 3e-7 * m_s_to_m_month_conversion,   # V11 was 0.1e-7 (Too low)

    # --- VOLCANIC DRIVERS (Sevier Headwaters) ---
    # Restoring the value that got Sevier to -15%
    'Extrusive igneous material': 8e-6 * m_s_to_m_month_conversion, # V11 was 0.5e-5
    'Intermediate-composition lava flows': 2e-5 * m_s_to_m_month_conversion,
    'Mafic-composition lava flows': 1e-5 * m_s_to_m_month_conversion,

    # --- BASIN FILL (Price Anchor) ---
    # Keeping these tight to protect Price
    'Sedimentary rock': 1e-10 * m_s_to_m_month_conversion,      # V11 was 5e-7 (Caused the Price spike)
    'Sandstone and mudstone': 1e-9 * m_s_to_m_month_conversion, # V11 was 5e-10

    # --- OTHERS (Standard V9) ---
    'Mostly sandstone': 5e-7 * m_s_to_m_month_conversion,
    'Sedimentary material': 1e-6 * m_s_to_m_month_conversion,
    'Alluvial sediment': 1e-4 * m_s_to_m_month_conversion,
    'Eolian sediment': 1e-4 * m_s_to_m_month_conversion,
    'Clastic sediment': 1e-5 * m_s_to_m_month_conversion,
    'Debris flows, landslides, and other localized mass-movement sediment': 1e-5 * m_s_to_m_month_conversion,
    'Peat and muck': 1e-5 * m_s_to_m_month_conversion,
    'Playa sediment': 1e-9 * m_s_to_m_month_conversion,
    'Medium and high-grade regional metamorphic rock, of unspecified origin': 1e-8 * m_s_to_m_month_conversion,
    'Metamorphic rock': 1e-9 * m_s_to_m_month_conversion,
    'Clastic sedimentary rock': 1e-6 * m_s_to_m_month_conversion,
    'Felsic-composition pyroclastic flows': 1e-6 * m_s_to_m_month_conversion,
    'Coarse-grained intrusive igneous rock': 1e-7 * m_s_to_m_month_conversion,
    'Coarse-grained, felsic-composition intrusive igneous rock': 1e-7 * m_s_to_m_month_conversion,
    'Coarse-grained, mafic-composition intrusive igneous rock': 1e-7 * m_s_to_m_month_conversion,
    'Intrusive igneous rock': 1e-7 * m_s_to_m_month_conversion,
    'Lacustrine sediment': 1e-9 * m_s_to_m_month_conversion,
    'Mostly carbonate rock': 1e-5 * m_s_to_m_month_conversion,
    'Sedimentary and extrusive igneous material': 1e-6 * m_s_to_m_month_conversion,
    'Water or ice': 1e-9 * m_s_to_m_month_conversion
}

def assign_k_value(feature):
    # Get the text description
    rock_type = feature.get('GeoMateria')
    
    # Earth Engine Dictionaries allow looking up values by string keys
    # We wrap the python dict into an ee.Dictionary
    gee_k_lookup = ee.Dictionary(k_values)
    
    # Get the value (Default to 0.0 if rock type not found)
    k_val = gee_k_lookup.get(rock_type, 0.0)
    
    # Set the new numeric property
    return feature.set('K_m_month', k_val)

# Map over the collection
geomaterials_with_k = geomaterials.map(assign_k_value)

In [4]:

def calculate_lithology_percentages(lithology_fc, basin_dict, output_path):
    """
    Calculates the percentage of area for each lithologic unit within named basin geometries.
    
    Args:
        lithology_fc (ee.FeatureCollection): The feature collection containing lithology data.
                                             Must have a property called 'GeoMateria'.
        basin_dict (dict): A dictionary where keys are basin names (str) and 
                           values are ee.Geometry objects.
                           Example: {'Basin_A': ee.Geometry(...), 'Basin_B': ...}
        output_path (str): The full file path (including .csv extension) for the output table.
    
    Returns:
        pd.DataFrame: A pandas DataFrame containing the results (also saved to CSV).
    """
    
    # 1. Validation
    if not isinstance(basin_dict, dict):
        raise ValueError("basin_dict must be a dictionary {name: ee.Geometry}.")
    
    # Check that all values are indeed geometries (or can be cast to them)
    for name, geom in basin_dict.items():
        if not isinstance(geom, ee.Geometry):
             # Try to handle cases where user might pass a Feature instead of Geometry
            if isinstance(geom, ee.Feature):
                basin_dict[name] = geom.geometry()
            else:
                raise ValueError(f"Value for key '{name}' is not a valid ee.Geometry or ee.Feature.")

    print(f"Processing {len(basin_dict)} basins...")
    
    results_dict = {}

    # 2. Iterate through the dictionary
    for basin_name, basin_geom in basin_dict.items():
        print(f"  Calculating statistics for: {basin_name}")
        
        # Calculate total basin area (maxError=1 for precision)
        total_basin_area = basin_geom.area(maxError=1)
        
        # Filter lithology to the specific basin bounds
        relevant_lithology = lithology_fc.filterBounds(basin_geom)
        
        # Define intersection logic
        def intersect_and_measure(feature):
            # Calculate intersection
            intersection = feature.geometry().intersection(basin_geom, ee.ErrorMargin(1))
            # Calculate area of the intersection
            intersection_area = intersection.area(maxError=1)
            # Return feature with new area property
            return feature.set({'intersect_area': intersection_area})

        # Map the intersection
        processed_fc = relevant_lithology.map(intersect_and_measure)
        
        # Group by 'GeoMateria' and sum 'intersect_area'
        # We filter > 0 to avoid processing non-intersecting features that passed the spatial filter
        stats = processed_fc.filter(ee.Filter.gt('intersect_area', 0)).reduceColumns(
            reducer=ee.Reducer.sum().group(
                groupField=1, 
                groupName='GeoMateria'
            ),
            selectors=['intersect_area', 'GeoMateria']
        )
        
        # Execute server-side operations
        try:
            client_stats = stats.get('groups').getInfo()
            client_total_area = total_basin_area.getInfo()
        except ee.EEException as e:
            print(f"    Error processing {basin_name}: {e}")
            results_dict[basin_name] = {}
            continue

        # Process results
        basin_data = {}
        if client_stats:
            for group in client_stats:
                material = group['GeoMateria']
                area_sqm = group['sum']
                
                # Calculate percentage
                pct = (area_sqm / client_total_area) * 100
                basin_data[material] = pct
        
        results_dict[basin_name] = basin_data

    # 3. Organize into DataFrame
    # Keys (Basin Names) become the columns
    df = pd.DataFrame.from_dict(results_dict, orient='index').T
    
    # Fill NaN with 0 and sort
    df = df.fillna(0)
    df = df.sort_index()

    # 4. Export
    output_dir = os.path.dirname(output_path)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    df.to_csv(output_path, index_label="GeoMateria")
    print(f"Success! Results saved to: {output_path}")
    
    return df

In [8]:
calculate_lithology_percentages(geomaterials_with_k, basin_geometries_dict, 'C:\\Users\\mradwin\\Documents\\Utah Soil Water Balance\\Basin_Lithology_Percentages_v12_GeoK.csv')

Processing 9 basins...
  Calculating statistics for: big_creek
  Calculating statistics for: blacks_fork
  Calculating statistics for: castle_creek
  Calculating statistics for: farmington_creek
  Calculating statistics for: mammoth_creek
  Calculating statistics for: muddy_creek
  Calculating statistics for: weber_river
  Calculating statistics for: west_canyon
  Calculating statistics for: whiterocks_uinta
Success! Results saved to: C:\Users\mradwin\Documents\Utah Soil Water Balance\Basin_Lithology_Percentages_v12_GeoK.csv


,big_creek,blacks_fork,castle_creek,mammoth_creek,muddy_creek,weber_river,west_canyon,whiterocks_uinta,farmington_creek
Alluvial sediment,7.162294,4.195260,33.504978,0.627816,0.523700,10.934553,13.149953,3.316347,0.000000
Clastic sediment,0.000000,0.000000,0.000000,0.796635,0.000000,0.000000,0.000000,0.000000,0.000000
Clastic sedimentary rock,0.408770,0.857441,0.000000,0.000000,44.389416,2.438548,0.000000,0.000000,0.000000
"Debris flows, landslides, and other localized mass-movement sediment",0.000000,0.000000,0.003330,0.000000,0.000000,0.445532,0.000000,0.264381,0.000000
Extrusive igneous material,0.000000,0.000000,0.000000,97.325171,0.000000,0.000000,0.000000,0.000000,0.000000
Felsic-composition pyroclastic flows,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.046556,0.000000,0.000000
Glacial till,0.000000,32.240834,0.300822,0.000000,0.004887,26.591232,0.000000,48.438024,13.908031
Intrusive igneous rock,0.000000,0.000000,13.089463,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
"Medium and high-grade regional metamorphic rock, of unspecified origin",0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,83.158926
Metasedimentary rock,3.133026,50.572871,0.000000,0.000000,0.000000,17.111670,0.000000,42.710849,0.000000


In [23]:
Map = geemap.Map(center=[39.5, -111.5], zoom=7)
test_col = ee.ImageCollection('projects/ut-gee-ugs-bsf-dev/assets/ModifiedUBM1TestingRuns/Mod_UBM_1_Testing_RF1kmST_POLPor_OLMFC_HHSWP_NGMDGeoKSdM_PRISMSNOM_ETGEESEBAL_IRRIm_T_M_m3')
test_col = GenericCollection(test_col)
print(test_col.image_grab(-1).bandNames().getInfo())
Map.addLayer(test_col.image_grab(-8), {'bands':['Soil_Water_End_Of_Previous_Timestep'], 'min':0, 'max':1e7, 'palette':get_palette('rdylbu')}, 'SoilWater')
for i, name in enumerate(basins_names):
    Map.addLayer(basins_list[i], {}, name)
Map

['precip_and_snowmelt_input', 'irrigation', 'AET', 'soil_thickness', 'soil_porosity', 'field_capacity', 'wilting_point', 'Geo_K', 'Runoff', 'Recharge', 'Soil_Water_End_Of_Previous_Timestep']


Map(center=[39.5, -111.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…